# Optical Flow Tracker

Compare sparse Lucas–Kanade feature tracking with dense Farneback optical flow on two video clips: moving shapes and ants. The pipeline exports trajectory overlays, direction-colored flow maps, binary motion masks, and per-video metrics.

Run all cells from the repository root after installing `requirements.txt`. The bundled inputs are `in_videos/shapes.mp4` and `in_videos/ants.mp4`; generated files go to `out/`.

- **Lucas–Kanade:** initialize Shi–Tomasi corners, track them between frames, and reject matches with high error or excessive displacement.
- **Farneback:** estimate motion at every pixel, visualize direction with hue, and threshold motion magnitude to extract moving regions.
- **Analysis:** compare surviving features, track lengths, flow magnitude, and motion-mask coverage. These are diagnostics, not ground-truth accuracy scores.

Change `VIDEO_LIST` and the settings below to process other clips. `MAX_FRAMES` includes the initial input frame, so the pipeline writes at most `MAX_FRAMES - 1` frames per output video. Motion is measured in pixels per frame after resizing. Output playback uses `FPS_OUT`, independently of the source frame rate.

In [ ]:
# Imports and configuration
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import deque

VIDEO_LIST = ["in_videos/shapes.mp4", "in_videos/ants.mp4"]
OUT_DIR = "out"
OUT_VIDEOS = os.path.join(OUT_DIR, "videos")
OUT_PICTURES = os.path.join(OUT_DIR, "pictures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(OUT_VIDEOS, exist_ok=True)
os.makedirs(OUT_PICTURES, exist_ok=True)

MAX_FRAMES, RESIZE_WIDTH, FPS_OUT = 150, 960, 30
GAUSS_BLUR, USE_CLAHE = (5, 5), True

GFTT_PARAMS = dict(maxCorners=500, qualityLevel=0.01, minDistance=7, blockSize=7)
LK_PARAMS = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
LK_ERR_THRESH, LK_MAX_JUMP = 20.0, 60.0
# Fraction of the initial feature count that triggers replenishment; 0 disables it.
REDETECT_PCT = 0.0

FB_PARAMS = dict(pyr_scale=0.5, levels=3, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
# Motion mask: magnitude threshold, morphological filtering, and minimum area
MAG_THRESH, MORPH_KERNEL, MIN_COMPONENT_AREA = 1.5, 5, 150


## Tracking and visualization helpers

In [ ]:
def resize_keep_aspect(frame, width):
    if width is None or frame.shape[1] == width:
        return frame
    h, w = frame.shape[:2]
    return cv2.resize(frame, (width, max(1, int(h * width / w))), interpolation=cv2.INTER_AREA)

def preprocess_gray(frame_bgr):
    frame_bgr = resize_keep_aspect(frame_bgr, RESIZE_WIDTH)
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    if GAUSS_BLUR:
        gray = cv2.GaussianBlur(gray, GAUSS_BLUR, 0)
    if USE_CLAHE:
        gray = cv2.createCLAHE(2.0, (8, 8)).apply(gray)
    return frame_bgr, gray


def flow_to_hsv_bgr(flow):
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0], hsv[..., 1] = (ang / 2).astype(np.uint8), 255
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR), mag

def clean_motion_mask(mask):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_KERNEL, MORPH_KERNEL))
    mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k, iterations=2)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = np.zeros_like(mask)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= MIN_COMPONENT_AREA:
            out[labels == i] = 255
    return out


TRACK_PALETTE = [
    (0, 255, 0), (0, 255, 255), (255, 0, 0), (255, 128, 0), (255, 0, 255),
    (0, 165, 255), (0, 128, 255), (128, 0, 255), (255, 255, 0), (128, 255, 0),
    (255, 0, 128), (0, 200, 200), (200, 0, 200), (200, 200, 0), (50, 205, 154),
    (255, 165, 0), (147, 20, 255), (255, 192, 203), (176, 224, 230), (255, 215, 0),
]
INITIAL_COLOR = (0, 255, 0)
REACQUIRED_PALETTE = [c for c in TRACK_PALETTE if c != INITIAL_COLOR] or [(0, 255, 255)]

def draw_lk_tracks(frame, tracks, track_colors):
    vis = frame.copy()
    for tr, color in zip(tracks, track_colors):
        if len(tr) < 2:
            continue
        pts = np.array(tr, dtype=np.int32)
        cv2.polylines(vis, [pts], False, color, 1, cv2.LINE_AA)
        cv2.circle(vis, tuple(pts[-1]), 4, (0, 0, 0), -1, cv2.LINE_AA)
    return vis

def track_lk_step(prev_gray, gray, tracks, alive, track_colors, redetect_thresh, new_pts_fn):
    """Advance active tracks and optionally append newly detected features."""
    idx = np.where(alive)[0]
    if len(idx) > 0:
        prev_pts = np.array([tracks[i][-1] for i in idx], dtype=np.float32).reshape(-1, 1, 2)
        next_pts, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, **LK_PARAMS)
        good = np.zeros(len(idx), dtype=bool)
        if next_pts is not None and st is not None and err is not None:
            good = st.ravel().astype(bool) & (err.ravel() < LK_ERR_THRESH)
            good &= np.isfinite(next_pts).all(axis=(1, 2))
            good &= np.linalg.norm((next_pts - prev_pts).reshape(-1, 2), axis=1) < LK_MAX_JUMP
        for j, i in enumerate(idx):
            if good[j]:
                tracks[i].append(tuple(next_pts[j, 0]))
            else:
                alive[i] = False
    if redetect_thresh > 0 and alive.sum() < redetect_thresh:
        pts = new_pts_fn(gray)
        n_add = min(200, len(pts))
        for pt in pts[:n_add]:
            tracks.append(deque([tuple(pt)], maxlen=MAX_FRAMES))
            track_colors.append(random.choice(REACQUIRED_PALETTE))
        alive = np.concatenate([alive, np.ones(n_add, dtype=bool)])
    return alive


## Video processing and exports

In [ ]:
def get_gftt_pts(g):
    p = cv2.goodFeaturesToTrack(g, mask=None, **GFTT_PARAMS)
    return p.reshape(-1, 2) if p is not None else np.empty((0, 2), dtype=np.float32)


def _open_video_and_first_frame(video_path):
    """Open video, read first frame; return (cap, frame_bgr, prev_gray, h, w) or (None,)*5."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Skip {video_path}: cannot open")
        cap.release()
        return None, None, None, None, None
    ret, frame0 = cap.read()
    if not ret:
        print(f"Skip {video_path}: empty")
        cap.release()
        return None, None, None, None, None
    try:
        frame_bgr, prev_gray = preprocess_gray(frame0)
    except Exception:
        cap.release()
        raise
    h, w = frame_bgr.shape[:2]
    return cap, frame_bgr, prev_gray, h, w


def _init_lk_tracks(prev_gray):
    """Initialize sparse tracks; a textureless frame can still use dense flow."""
    points = get_gftt_pts(prev_gray)
    tracks = [deque([tuple(pt)], maxlen=MAX_FRAMES) for pt in points]
    track_colors = [INITIAL_COLOR] * len(tracks)
    alive = np.ones(len(points), dtype=bool)
    return tracks, track_colors, alive, len(tracks)


def _create_video_writers(out_v, w, h):
    """Open the LK overlay, flow visualization, and binary-mask video outputs."""
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writers = {}
    try:
        for key, filename in [
            ("lk", "lk_tracks.avi"),
            ("flow", "farneback_flow.avi"),
            ("mask", "motion_mask.avi"),
        ]:
            path = os.path.join(out_v, filename)
            writer = cv2.VideoWriter(path, fourcc, FPS_OUT, (w, h))
            writers[key] = writer
            if not writer.isOpened():
                raise RuntimeError(f"Cannot open video writer: {path}")
    except Exception:
        for writer in writers.values():
            writer.release()
        raise
    return writers


def _lk_frame_metrics(tracks, alive, jump_thr=25.0):
    """From current tracks/alive: mean displacement and jump ratio (displacement > jump_thr) per alive point."""
    disps = []
    jumps = 0
    for i in range(len(tracks)):
        if not alive[i] or len(tracks[i]) < 2:
            continue
        d = float(np.linalg.norm(np.array(tracks[i][-1]) - np.array(tracks[i][-2])))
        disps.append(d)
        if d > jump_thr:
            jumps += 1
    n_alive = max(1, int(alive.sum()))
    mean_disp = float(np.mean(disps)) if disps else 0.0
    jump_ratio = jumps / n_alive
    return mean_disp, jump_ratio


def _flow_and_mask_frame(prev_gray, gray):
    """Farneback flow + magnitude/mask stats. Returns (flow_bgr, mean_mag, p95_mag, mask, mask_area_ratio, n_components)."""
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, **FB_PARAMS)
    flow_bgr, mag = flow_to_hsv_bgr(flow)
    mask = clean_motion_mask((mag > MAG_THRESH).astype(np.uint8) * 255)
    mean_mag = float(np.mean(mag))
    p95_mag = float(np.percentile(mag, 95))
    mask_area_ratio = (mask > 0).sum() / mask.size
    n_cc, _, _, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    return flow_bgr, mean_mag, p95_mag, mask, mask_area_ratio, n_cc - 1


def _write_frame_videos(writers, frame_bgr, tracks, track_colors, n_initial, n_alive, frame_idx, flow_bgr, mask):
    """Draw LK overlay with text, then write lk, flow, mask frames to writers."""
    pct = 100.0 * n_alive / n_initial if n_initial else 0
    txt = f"Frame {frame_idx+2}  Init: {n_initial}  Alive: {n_alive}  {pct:.1f}%"
    lk_vis = draw_lk_tracks(frame_bgr, tracks, track_colors)
    cv2.putText(lk_vis, txt, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(lk_vis, txt, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    writers["lk"].write(lk_vis)
    writers["flow"].write(flow_bgr)
    writers["mask"].write(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))


def _process_frames(cap, writers, tracks, alive, track_colors, n_initial, prev_gray):
    """Process consecutive frame pairs and return their count and diagnostics."""
    JUMP_THR = 25.0
    redetect_thresh = max(1, int(REDETECT_PCT * n_initial)) if REDETECT_PCT > 0 else 0
    survival_rates, mean_track_lens = [], []
    mask_area_ratios, mask_num_components = [], []
    alive_counts, mean_disp_list, jump_ratio_list, mean_mag_list, p95_mag_list = [], [], [], [], []
    frame_count = 0
    for _ in range(MAX_FRAMES - 1):
        ret, frame = cap.read()
        if not ret:
            break
        frame_bgr, gray = preprocess_gray(frame)
        alive = track_lk_step(prev_gray, gray, tracks, alive, track_colors, redetect_thresh, get_gftt_pts)
        n_alive = int(alive.sum())
        alive_counts.append(n_alive)
        mean_disp, jump_ratio = _lk_frame_metrics(tracks, alive, JUMP_THR)
        mean_disp_list.append(mean_disp)
        jump_ratio_list.append(jump_ratio)
        flow_bgr, mean_mag, p95_mag, mask, mask_area_ratio, n_cc = _flow_and_mask_frame(prev_gray, gray)
        mean_mag_list.append(mean_mag)
        p95_mag_list.append(p95_mag)
        mask_area_ratios.append(mask_area_ratio)
        mask_num_components.append(n_cc)
        _write_frame_videos(writers, frame_bgr, tracks, track_colors, n_initial, n_alive, frame_count, flow_bgr, mask)
        survival_rates.append(alive.sum() / max(1, len(alive)))
        lengths = [len(tracks[i]) for i in range(len(tracks)) if alive[i]]
        mean_track_lens.append(np.mean(lengths) if lengths else 0.0)
        prev_gray = gray
        frame_count += 1
    return (
        frame_count,
        alive_counts, survival_rates, mean_track_lens,
        mask_area_ratios, mask_num_components,
        mean_disp_list, jump_ratio_list, mean_mag_list, p95_mag_list,
        alive,
    )


def _write_metrics_and_plot(name, out_dir, out_p, metrics, alive_counts, n_initial):
    """Write metrics txt and save survival plot."""
    with open(os.path.join(out_dir, f"{name}_metrics.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(f"{k}: {v}" for k, v in metrics.items()))
    fig, ax = plt.subplots(figsize=(8, 4))
    frames = np.arange(2, len(alive_counts) + 2)
    ax.plot(frames, alive_counts, label="Alive")
    ax.axhline(n_initial, color="gray", linestyle="--", label="Initial")
    ax.set_xlabel("Source frame")
    ax.set_ylabel("Points")
    ax.set_title(f"Track survival — {name}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(os.path.join(out_p, "metrics_by_frame.png"), dpi=100, bbox_inches="tight")
    plt.close(fig)


def _save_sample_frames(out_v, out_p):
    """Save the first, middle, and last available output frames."""
    for name in ["lk_tracks", "farneback_flow", "motion_mask"]:
        cap_out = cv2.VideoCapture(os.path.join(out_v, f"{name}.avi"))
        try:
            if not cap_out.isOpened():
                continue
            n_frames = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))
            if n_frames == 0:
                continue
            for idx in sorted({0, n_frames // 2, n_frames - 1}):
                cap_out.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ok, img = cap_out.read()
                if ok:
                    cv2.imwrite(os.path.join(out_p, f"{name}_frame{idx}.png"), img)
        finally:
            cap_out.release()


In [ ]:
def run_pipeline(video_path):
    """Export LK tracks, dense flow, motion masks, metrics, and sample images."""
    if MAX_FRAMES < 1:
        raise ValueError("MAX_FRAMES must be at least 1")
    if not 0 <= REDETECT_PCT <= 1:
        raise ValueError("REDETECT_PCT must be between 0 and 1")
    name = os.path.splitext(os.path.basename(video_path))[0]
    out_v = os.path.join(OUT_VIDEOS, name)
    out_p = os.path.join(OUT_PICTURES, name)
    os.makedirs(out_v, exist_ok=True)
    os.makedirs(out_p, exist_ok=True)

    cap, _, prev_gray, h, w = _open_video_and_first_frame(video_path)
    if cap is None:
        return None

    writers = {}
    try:
        tracks, track_colors, alive, n_initial = _init_lk_tracks(prev_gray)
        writers = _create_video_writers(out_v, w, h)
        (
            frame_count,
            alive_counts, survival_rates, mean_track_lens,
            mask_area_ratios, mask_num_components,
            mean_disp_list, jump_ratio_list, mean_mag_list, p95_mag_list,
            alive,
        ) = _process_frames(cap, writers, tracks, alive, track_colors, n_initial, prev_gray)
    finally:
        cap.release()
        for writer in writers.values():
            writer.release()

    def mean_or_none(values):
        return float(np.mean(values)) if values else None

    metrics = {
        "frames": frame_count,
        "input_frames": frame_count + 1,
        "alive": f"{int(alive.sum())}/{len(alive)}",
        "survival_rate": mean_or_none(survival_rates),
        "mean_track_len": mean_or_none(mean_track_lens),
        "mask_area_ratio": mean_or_none(mask_area_ratios),
        "mask_components": mean_or_none(mask_num_components),
    }
    _write_metrics_and_plot(name, OUT_DIR, out_p, metrics, alive_counts, n_initial)
    if frame_count:
        _save_sample_frames(out_v, out_p)
    else:
        print(f"{name}: no consecutive frame pairs available for optical flow")

    print(f"{name}: {frame_count} output frames -> {out_v}")
    return {
        "metrics": metrics, "name": name, "n_initial": n_initial,
        "alive_counts": alive_counts, "survival_rates": survival_rates,
        "mean_disp": mean_disp_list, "jump_ratio": jump_ratio_list,
        "mean_mag": mean_mag_list, "p95_mag": p95_mag_list,
        "mask_components": mask_num_components, "mask_area_ratios": mask_area_ratios,
        "frame_count": frame_count, "out_v": out_v, "tracks": tracks, "height": h,
    }


## Run the pipeline

In [ ]:
# Process the configured input videos
results = {}
for video in VIDEO_LIST:
    name = os.path.splitext(os.path.basename(video))[0]
    results[name] = run_pipeline(video)


## Inspect trajectories and motion diagnostics

In [ ]:
# Choose source-frame numbers for the detailed views.
ANTS_FRAME_TO_SHOW = 44
SHAPES_FRAME_TO_SHOW = 75


def plot_tracks_xy(tracks, height, title="LK trajectories in frame coordinates"):
    """Plot feature paths in image coordinates, with the origin at the top left."""
    fig, ax = plt.subplots(figsize=(7, 6))
    for track in tracks:
        if len(track) >= 2:
            xy = np.array(track)
            ax.plot(xy[:, 0], xy[:, 1], linewidth=1)
    ax.set_xlim(left=0)
    ax.set_ylim(height, 0)
    ax.set_title(title)
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
    ax.grid(True, alpha=0.3)
    plt.show()
    plt.close(fig)


def _read_output_frame(path, index):
    """Read one frame while releasing the capture on every path."""
    cap = cv2.VideoCapture(path)
    try:
        if not cap.isOpened():
            return None
        cap.set(cv2.CAP_PROP_POS_FRAMES, index)
        ok, frame = cap.read()
        return frame if ok else None
    finally:
        cap.release()


def _run_analysis_plots(name, result, frame_to_show, sample_frames_only=False):
    """Show sample outputs and optional sparse/dense motion diagnostics."""
    frame_count = result["frame_count"]
    if not frame_count:
        print(f"{name}: no output frames to plot")
        return
    if not sample_frames_only:
        frames = np.arange(2, len(result["alive_counts"]) + 2)
        index = max(0, min(frame_to_show - 2, frame_count - 1))
        image = _read_output_frame(os.path.join(result["out_v"], "lk_tracks.avi"), index)
        if image is not None:
            n_alive = result["alive_counts"][index]
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{name}: source frame {index + 2} (active: {n_alive}, initial: {result['n_initial']})")
            ax.axis("off")
            plt.show()
            plt.close(fig)
        plot_tracks_xy(result["tracks"], result["height"], title=f"{name}: LK trajectories")
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        ax = axes[0, 0]
        ax.plot(frames, result["alive_counts"], label="Active")
        ax.axhline(result["n_initial"], color="gray", linestyle="--", label="Initial")
        ax.set_ylabel("Points")
        ax.set_title("LK: active feature count")
        ax.legend()
        ax = axes[0, 1]
        ax.plot(frames, np.array(result["survival_rates"]) * 100)
        ax.set_ylabel("Active tracks (%)")
        ax.set_title("LK: active fraction of all detected tracks")
        ax = axes[1, 0]
        ax.plot(frames, result["mean_mag"], label="Mean")
        ax.plot(frames, result["p95_mag"], label="95th percentile")
        ax.set_ylabel("Pixels per frame")
        ax.set_title("Farneback: flow magnitude")
        ax.legend()
        ax = axes[1, 1]
        ax.plot(frames, np.array(result["mask_area_ratios"]) * 100)
        ax.set_ylabel("Area (%)")
        ax.set_title("Farneback: motion-mask coverage")
        for ax in axes.ravel():
            ax.set_xlabel("Source frame")
            ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()
        plt.close(fig)

    sample_indices = [min(i, frame_count - 1) for i in [0, frame_count // 4, frame_count // 2, 3 * frame_count // 4]]
    for filename, grayscale, title in [
        ("lk_tracks.avi", False, "LK tracks"),
        ("farneback_flow.avi", False, "Farneback flow HSV"),
        ("motion_mask.avi", True, "Motion mask"),
    ]:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        for ax, index in zip(axes.ravel(), sample_indices):
            image = _read_output_frame(os.path.join(result["out_v"], filename), index)
            if image is None:
                ax.text(0.5, 0.5, "Frame unavailable", ha="center", va="center", transform=ax.transAxes)
            elif grayscale:
                ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), cmap="gray")
            else:
                ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{title}: source frame {index + 2}")
            ax.axis("off")
        fig.tight_layout()
        plt.show()
        plt.close(fig)


if results.get("shapes") is not None:
    _run_analysis_plots("Shapes", results["shapes"], SHAPES_FRAME_TO_SHOW, sample_frames_only=True)
if results.get("ants") is not None:
    _run_analysis_plots("Ants", results["ants"], ANTS_FRAME_TO_SHOW)


## Interpreting the results

**Shapes** provides a controlled scene with high-contrast boundaries. Inspect how well the sparse trajectories follow corners and how the dense flow concentrates around visible edges. Uniform interiors provide little local texture, so a motion mask need not fill an entire moving shape.

**Ants** provides a more demanding scene with overlapping objects, small moving limbs, changing appearance, and objects leaving the frame. Inspect drops in active feature count alongside the overlays. A surviving feature can still drift; survival alone does not establish tracking accuracy.

### Sparse tracking

All initial tracks are green. Their histories remain visible after a track is lost. When `REDETECT_PCT` is positive, replacement features receive other colors. Replenishment adds fresh tracks without recovering the identities of lost ones or assigning identities to individual objects.

With redetection disabled, the active fraction measures how many initial features remain. With redetection enabled, its denominator includes every feature detected so far. `mean_track_len` averages the lengths of currently active tracks at each processed frame, then averages those values across the clip. An overlay's active-to-initial percentage can exceed 100% when new features are added.

### Dense flow and motion masks

Hue represents flow direction. Brightness represents magnitude normalized separately for each frame, so brightness cannot be compared quantitatively across frames. Use the mean and 95th-percentile magnitude curves for that comparison.

The binary motion mask thresholds flow magnitude, applies morphological filtering, and removes small connected components. It identifies estimated motion, not individual objects: nearby moving regions can merge, while different parts of one object can form separate regions. Camera motion and noisy flow can also appear in the mask.

### Metrics and limitations

`frames` is the number of processed frame pairs and written output frames; `input_frames` includes the initial reference frame. Output frame 0 corresponds to source frame 2. Summary rates and mask statistics are time averages over those pairs. Videos without a frame pair have no mean statistics (`None`).

The LK jump diagnostic counts accepted displacements above 25 pixels per frame; larger jumps rejected by `LK_MAX_JUMP` are absent from that diagnostic. Both magnitudes and thresholds depend on resizing and the source frame rate. The mask threshold is therefore not a speed in physical units.

Use Lucas–Kanade when feature trajectories are useful and Farneback when a field of motion is useful. Neither method in this notebook performs object detection, preserves object identity through occlusion, or evaluates error against labeled ground truth. Rerun the notebook to obtain metrics for the current parameters instead of relying on fixed benchmark values.